# Problemas detectados en versiones anteriores y cómo se resolvieron

TweetAge v6: modelos de lenguaje de caracteres y palabras, Kneser-Ney modificado y particion por texto normalizado.

| Problema | Evidencia | Solución en v6 | Base |
|---|---|---|---|
| Vectorización prohibida en v1-v4 | Analisis previo: TweetAge_modelo, lineas JSON 731/733; v2, 49; v3, 74/120/137; v4, 57/74 | Sin vectorizadores; guardia AST | Enunciado |
| Fuga entre desarrollo y holdout | v5 reporta 2.561 textos normalizados del holdout presentes en desarrollo | Grupos de texto normalizado; interseccion verificada en cero | Enunciado |
| Split historico no certificable | Sin indices originales de Kelly en el repo | Indices, grupos y SHA-256 del train guardados | Enunciado |
| Seleccion y comparacion sobre el mismo holdout | Script v3 compara cuatro modelos en nivel3_evaluar | Seleccion en CV dentro de desarrollo; holdout una sola vez | Práctica estándar |
| Score sin normalizar por longitud | v5 usa score bruto y tiene F1 distintos por clase; eso no prueba causalidad por longitud | Contrastar score bruto y dividido por longitud en CV | Decisión propia sin fuente |
| F1 OOF optimista tras calibrar | En v5, el ajuste de b usa las mismas etiquetas OOF que el score | Reportar antes y despues; mantener holdout independiente | Práctica estándar |
| Confusion entre holdout y Kaggle | v3: 0.6075 de holdout y 0.61765 de Kaggle, conjuntos distintos | Reportar separados; no tratarlos como mediciones emparejadas | Práctica estándar |
| Entorno temporal en /tmp | Ejecucion de v5 en entorno temporal | Entorno .venv-v6 dentro del proyecto y requirements.txt fijado | Enunciado |
| Nombres y separadores incorrectos | test.csv/submission.csv no existen en los datos suministrados | age_test.csv con punto y coma y BOM; salida con coma sin BOM | Enunciado |
| Nota de v4 desactualizada | v4 dice por medir; el usuario comunica Kaggle 0.62832, no verificado independientemente | Anotar este dato con su procedencia, sin modificar v4 | Decisión propia sin fuente |
| Descuento unico de v5 | D=0.75 constante en v5 | D1, D2 y D3+ estimados por clase, orden y tipo de conteo | Literatura (Chen y Goodman, 1999) |

Las referencias a enunciado p. 2, criterios minimos p. 3 y reproducibilidad proceden del requerimiento del usuario; no se dispone aqui del documento para verificar sus paginas. Las bases se restringen a las cuatro categorias solicitadas; definicion de metrica y formato se clasifican como practica estandar y enunciado, respectivamente. El dato 0.62832 no se presenta como una consulta realizada al leaderboard.

Referencia: Chen, S. F., y Goodman, J. (1999). An empirical study of smoothing techniques for language modeling. Computer Speech & Language, 13(4), 359-394. Los umbrales de vocabulario, el fallback de descuentos y la division concreta por longitud son decisiones propias descritas abajo.

In [1]:
import ast
import csv
import gc
import hashlib
import html
import json
import math
import platform
import re
import time
from collections import Counter
from functools import lru_cache
from itertools import product
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
CHAR_ORDERS, WORD_ORDERS = [3, 4, 5, 6], [1, 2, 3]
LAMBDAS = [0.0, 0.25, 0.5, 0.75, 1.0]
SCORE_MODES = ['bruto', 'longitud']
BASE_EPSILON = 1e-6
TIME_LIMIT = 1200
CLASSES = np.array(['13-17', '18-24', '25-34', '35-49', '50-64', '65-xx'])
BOS, EOS, UNK = '\x02', '\x03', '\x01'
roots = [Path.cwd(), Path.cwd() / 'workshops' / 'Kaggle-tweet-age', Path.cwd() / 'natural-language-processing-workshops' / 'workshops' / 'Kaggle-tweet-age']
OUT_DIR = next((candidate.resolve() for candidate in roots if (candidate / 'TweetAge_v6_charLM_kn.ipynb').is_file()), None)
assert OUT_DIR is not None, 'Ejecutar desde la carpeta del notebook o la raiz del repositorio.'
NOTEBOOK_PATH = OUT_DIR / 'TweetAge_v6_charLM_kn.ipynb'
SPLIT_PATH = OUT_DIR / 'split_idx_v6_seed42.npz'
SUB_PATH = OUT_DIR / 'age_submission_charLM_kn_v6.csv'
DATA_DIR = OUT_DIR / 'tweet-age-2026-puj'
assert not SPLIT_PATH.exists() and not SUB_PATH.exists(), 'Se detiene para no sobrescribir salidas existentes.'
INPUT_HASHES = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in ['age_train.csv', 'age_test.csv', 'age_submission.csv']}
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__, '| sklearn', sklearn.__version__)
print('Semilla', SEED, '| Notebook', NOTEBOOK_PATH)

def check_compliance(notebook):
    forbidden = {'tfidfvectorizer', 'countvectorizer', 'hashingvectorizer', 'dictvectorizer', 'feature_extraction', 'linear_model', 'transformers', 'transformer', 'embedding', 'embeddings', 'word2vec', 'glove'}
    problems = []
    for cell_number, cell in enumerate(notebook['cells'], 1):
        if cell['cell_type'] != 'code':
            continue
        source = ''.join(cell['source']) if isinstance(cell['source'], list) else cell['source']
        for node in ast.walk(ast.parse(source)):
            names = []
            if isinstance(node, ast.Name):
                names = [node.id]
            elif isinstance(node, ast.Attribute):
                names = [node.attr]
            elif isinstance(node, ast.Import):
                names = [alias.name for alias in node.names]
            elif isinstance(node, ast.ImportFrom):
                names = [node.module or ''] + [alias.name for alias in node.names]
            for name in names:
                if any(part.lower() in forbidden for part in name.split('.')):
                    problems.append((cell_number, node.lineno, name))
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in {'eval', 'exec', '__import__'}:
                problems.append((cell_number, node.lineno, 'ejecucion dinamica'))
    if problems:
        raise RuntimeError(f'Guardia AST: codigo prohibido {problems}')
    return True

with NOTEBOOK_PATH.open(encoding='utf-8') as handle:
    assert check_compliance(json.load(handle))
print('Guardia AST superada; comentarios, Markdown y literales de la propia guardia no se confunden con usos.')

Python 3.12.14 | numpy 2.5.4 | pandas 3.0.6 | sklearn 1.9.1
Semilla 42 | Notebook ./workshops/Kaggle-tweet-age/TweetAge_v6_charLM_kn.ipynb
Guardia AST superada; comentarios, Markdown y literales de la propia guardia no se confunden con usos.


## Datos, normalizacion y grupos

Se decodifica HTML y se excluyen textos vacios o con `#¿NOMBRE?`. URLs pasan a `<url>`, menciones a `<user>` y repeticiones de letras a un maximo de tres, conservando mayusculas en caracteres. El tokenizador de palabras pasa a minusculas. Signos y emojis se conservan; se retiene tambien la puntuacion final adyacente a las URLs como heuristica propia.

Los grupos se definen por el texto normalizado identico, conservando mayusculas. Si un grupo tiene varias etiquetas, su estrato es la clase mayoritaria, con desempate por orden de edad. Se estratifican los grupos, no las filas: el 80/20 es de grupos y la proporcion de filas puede variar. Se informan conflictos y distribuciones. Los cinco pliegues internos tambien se construyen sobre grupos con StratifiedKFold y se expanden a filas; ninguna repeticion identica cruza un pliegue. Esta es una decision explicita para conciliar estratificacion y grupos con etiquetas contradictorias.

In [2]:
def verify_csv(path, columns, delimiter, bom):
    with path.open('rb') as handle:
        prefix = handle.read(32768)
    first_line = prefix.decode('utf-8-sig').splitlines()[0]
    actual = csv.Sniffer().sniff(first_line, delimiters=',;\t|').delimiter
    header = next(csv.reader([first_line], delimiter=actual))
    assert actual == delimiter and prefix.startswith(b'\xef\xbb\xbf') == bom, (path.name, actual)
    assert header == columns, (path.name, header)
    print(path.name, 'BOM=', bom, 'separador=', repr(actual), 'columnas=', header)
    return header

verify_csv(DATA_DIR / 'age_train.csv', ['id', 'text', 'age_range'], ';', True)
reference_columns = verify_csv(DATA_DIR / 'age_submission.csv', ['id', 'age_range'], ',', False)
train_raw = pd.read_csv(DATA_DIR / 'age_train.csv', sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert list(train_raw.columns) == ['id', 'text', 'age_range']
clean_text = train_raw.text.fillna('').str.strip().map(html.unescape)
excluded = clean_text.eq('') | clean_text.str.contains('#\u00bfNOMBRE', regex=False)
train = train_raw.loc[~excluded].copy()
train['raw_row'] = np.flatnonzero(~excluded.to_numpy())
train['text'] = clean_text.loc[~excluded]
train = train.reset_index(drop=True)
assert train.id.is_unique and set(train.age_range) == set(CLASSES)
URL_RE = re.compile(r'https?://[^\s]+|www\.[^\s]+', re.IGNORECASE)
MENTION_RE = re.compile(r'@\w+', re.UNICODE)
REPEAT_RE = re.compile(r'([^\W\d_])\1{3,}', re.UNICODE)

def replace_url(match):
    value = match.group(0)
    core = value.rstrip('.,!?;:)]}')
    return '<url>' + value[len(core):]

def normalize_text(text):
    return REPEAT_RE.sub(lambda match: match.group(1) * 3, MENTION_RE.sub('<user>', URL_RE.sub(replace_url, text)))

assert normalize_text('Hoooooola!!!! @Ana https://a.co).') == 'Hooola!!!! <user> <url>).'
assert normalize_text('JAJA \U0001f600\U0001f600 !!!!') == 'JAJA \U0001f600\U0001f600 !!!!'
train['normalized'] = train.text.map(normalize_text)
y_all = pd.Categorical(train.age_range, categories=CLASSES).codes.astype(np.int64)
group_ids, unique_texts = pd.factorize(train.normalized, sort=True)
group_counts = np.zeros((len(unique_texts), 6), dtype=np.int64)
np.add.at(group_counts, (group_ids, y_all), 1)
group_targets = group_counts.argmax(axis=1)
conflicting_groups = int(((group_counts > 0).sum(axis=1) > 1).sum())
dev_groups, holdout_groups = train_test_split(np.arange(len(unique_texts)), test_size=0.2, stratify=group_targets, random_state=SEED)
idx_development = np.flatnonzero(np.isin(group_ids, dev_groups))
idx_holdout = np.flatnonzero(np.isin(group_ids, holdout_groups))
intersection = set(train.normalized.iloc[idx_development]) & set(train.normalized.iloc[idx_holdout])
assert not intersection and not np.intersect1d(dev_groups, holdout_groups).size
assert len(idx_development) + len(idx_holdout) == len(train)
with SPLIT_PATH.open('xb') as handle:
    np.savez_compressed(handle, train_idx=idx_development, valid_idx=idx_holdout, raw_train_idx=train.raw_row.to_numpy()[idx_development], raw_valid_idx=train.raw_row.to_numpy()[idx_holdout], retained_raw_idx=train.raw_row.to_numpy(), group_ids=group_ids, train_groups=dev_groups, valid_groups=holdout_groups, seed=np.array(SEED), train_sha256=np.array(INPUT_HASHES['age_train.csv']))
texts_development = train.normalized.iloc[idx_development].to_numpy()
y_development = y_all[idx_development]
groups_development = group_ids[idx_development]
lengths_development = np.array([max(1, len(text)) for text in texts_development], dtype=float)
split_table = pd.DataFrame({'lado': ['desarrollo', 'holdout'], 'grupos': [len(dev_groups), len(holdout_groups)], 'filas': [len(idx_development), len(idx_holdout)]})
print('Originales:', len(train_raw), '| Excluidos:', int(excluded.sum()), '| Limpios:', len(train))
print('Solo <url>:', int(train.normalized.str.strip().eq('<url>').sum()), '| Grupos con etiquetas contradictorias:', conflicting_groups)
print('Interseccion de textos normalizados:', len(intersection))
display(split_table)
display(pd.DataFrame({'train_limpio': train.age_range.value_counts().reindex(CLASSES), 'desarrollo': train.age_range.iloc[idx_development].value_counts().reindex(CLASSES), 'holdout': train.age_range.iloc[idx_holdout].value_counts().reindex(CLASSES)}))

age_train.csv BOM= True separador= ';' columnas= ['id', 'text', 'age_range']
age_submission.csv BOM= False separador= ',' columnas= ['id', 'age_range']


Originales: 169336 | Excluidos: 155 | Limpios: 169181
Solo <url>: 3785 | Grupos con etiquetas contradictorias: 680
Interseccion de textos normalizados: 0


,lado,grupos,filas
0,desarrollo,126232,133065
1,holdout,31558,36116


,train_limpio,desarrollo,holdout
age_range,,,
13-17,28785,22859,5926
18-24,28626,22627,5999
25-34,28324,22399,5925
35-49,28226,22219,6007
50-64,28060,21959,6101
65-xx,27160,21002,6158


## Kneser-Ney modificado

En cada clase y orden se calculan los conteos de conteos N1, N2, N3, N4. Y=N1/(N1+2N2); D1=1-2Y N2/N1; D2=2-3Y N3/N2; D3+=3-4Y N4/N3. El orden superior usa frecuencias y los inferiores, numero de predecesores distintos. La masa de retroceso de un historial es la suma de los descuentos realmente retirados, dividida entre el total del historial; no se aproxima mediante un descuento unico.

Si faltan denominadores se usa 0.75 como fallback conservador por bucket (decision propia); se limita cada descuento al intervalo (0,r) correspondiente. La base es la distribucion de continuaciones de bigramas con epsilon=1e-6 para soporte positivo normalizado. El modelo de palabras de orden 1 tambien usa esa base. BOS nunca se predice; cada texto acaba en EOS. Palabras de frecuencia menor que dos en el ajuste pasan a UNK; el vocabulario es compartido entre clases pero exclusivo del ajuste de cada pliegue. Caracteres usa los observados y un UNK para caracteres nuevos.

Score bruto = lambda log P_car + (1-lambda) log P_pal + log prior. La variante por longitud divide este score completo (incluido prior) por max(1, numero de caracteres del texto normalizado), y despues suma el sesgo b. Esta definicion de longitud es una decision propia, no una exigencia de Kneser-Ney. No se normaliza cada modelo por una longitud diferente.

Se construyen los conteos hasta el mayor orden una vez por clase y pliegue. Los tiempos compartidos se reparten entre ordenes para informar el coste amortizado, y se registran aparte. Lambda, variante y sesgos se calculan a partir de esas mismas puntuaciones; no implican nuevos entrenamientos.

In [3]:
TOKEN_RE = re.compile(r'<url>|<user>|\w+|[^\w\s]', re.UNICODE)

def tokenize(text, level):
    return list(text) if level == 'char' else TOKEN_RE.findall(text.lower())

def make_encoder(texts, level):
    frequencies = Counter()
    for text in texts:
        frequencies.update(tokenize(text, level))
    cutoff = 1 if level == 'char' else 2
    return {token: chr(index + 256) for index, token in enumerate(sorted(token for token, count in frequencies.items() if count >= cutoff))}

def encode_text(text, level, mapping):
    return ''.join(mapping.get(token, UNK) for token in tokenize(text, level))

def estimate_discounts(counts):
    histogram = Counter(counts.values())
    count1, count2, count3, count4 = [histogram.get(value, 0) for value in [1, 2, 3, 4]]
    factor = count1 / (count1 + 2 * count2) if count1 + 2 * count2 else 0.0
    first = 1 - 2 * factor * count2 / count1 if count1 and count2 else 0.75
    second = 2 - 3 * factor * count3 / count2 if count1 and count2 else 0.75
    third = 3 - 4 * factor * count4 / count3 if count1 and count2 and count3 else 0.75
    values = tuple(float(min(bucket - 1e-6, max(1e-6, value))) for bucket, value in enumerate([first, second, third], 1))
    return values, dict(N1=count1, N2=count2, N3=count3, N4=count4)

def discount_for(count, discounts):
    return discounts[min(count, 3) - 1] if count > 0 else 0.0

class ModifiedKN:
    def __init__(self, max_order, support):
        self.max_order = max_order
        self.support = set(support) | {EOS, UNK}

    def fit(self, sequences):
        highest = max(2, self.max_order)
        self.raw = {order: Counter() for order in range(2, highest + 1)}
        for sequence in sequences:
            padded = BOS * (highest - 1) + sequence + EOS
            for order, counts in self.raw.items():
                counts.update(padded[position - order + 1:position + 1] for position in range(highest - 1, len(padded)))
        self.continuation = {}
        for order in range(1, highest):
            effective = Counter()
            for gram in self.raw[order + 1]:
                effective[gram[1:]] += 1
            self.continuation[order] = effective
        self.base = self.continuation[1]
        self.base_total = sum(self.base.values()) + BASE_EPSILON * len(self.support)
        self.discount_records = []
        def prepare(count_tables, origin):
            discounts_by_order, histories_by_order = {}, {}
            for order, counts in count_tables.items():
                if order == 1:
                    continue
                discounts, histogram = estimate_discounts(counts)
                discounts_by_order[order] = discounts
                totals, removed = Counter(), Counter()
                for gram, count in counts.items():
                    totals[gram[:-1]] += count
                    removed[gram[:-1]] += discount_for(count, discounts)
                histories_by_order[order] = {history: (total, removed[history]) for history, total in totals.items()}
                self.discount_records.append(dict(origen=origin, orden=order, D1=discounts[0], D2=discounts[1], D3=discounts[2], **histogram))
            return discounts_by_order, histories_by_order
        self.raw_discounts, self.raw_hist = prepare(self.raw, 'frecuencias')
        self.cont_discounts, self.cont_hist = prepare(self.continuation, 'continuaciones')
        self.lower_cached = lru_cache(maxsize=200000)(self._lower)
        self.log_cached = lru_cache(maxsize=200000)(self._log_gram)
        return self

    def _lower(self, gram):
        if len(gram) == 1:
            return (self.base.get(gram, 0) + BASE_EPSILON) / self.base_total
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.cont_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.continuation[order].get(gram, 0)
        return (count - discount_for(count, self.cont_discounts[order]) + removed * lower) / total

    def probability(self, gram):
        if len(gram) == 1:
            return self.lower_cached(gram)
        order = len(gram)
        lower = self.lower_cached(gram[1:])
        total, removed = self.raw_hist[order].get(gram[:-1], (0, 0.0))
        if not total:
            return lower
        count = self.raw[order].get(gram, 0)
        return (count - discount_for(count, self.raw_discounts[order]) + removed * lower) / total

    def _log_gram(self, gram):
        probability = self.probability(gram)
        if not 0 < probability <= 1 + 1e-10:
            raise ArithmeticError((repr(gram), probability))
        return math.log(probability)

    def log_probability(self, sequence, order):
        assert 1 <= order <= self.max_order
        padded = BOS * (order - 1) + sequence + EOS
        grams = Counter(padded[position - order + 1:position + 1] for position in range(order - 1, len(padded)))
        return sum(count * self.log_cached(gram) for gram, count in grams.items())

    def close(self):
        self.lower_cached.cache_clear()
        self.log_cached.cache_clear()

def test_modified_kn():
    values, _ = estimate_discounts(Counter({'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 1, 'f': 2, 'g': 3, 'h': 1}))
    assert len(set(values)) == 3 and all(0 < value < bucket for bucket, value in enumerate(values, 1))
    model = ModifiedKN(4, set('abc')).fit(['ab', 'ab', 'ac', 'bc', 'abc', 'abc', 'abc'])
    assert model.base['a'] == 1 and model.raw[2][BOS + 'a'] == 6
    for order in [1, 2, 3, 4]:
        contexts = set(model.raw_hist.get(order, {})) | {'', 'zz', BOS * (order - 1)}
        for context in contexts:
            history = (BOS * (order - 1) + context)[-(order - 1):] if order > 1 else ''
            assert abs(sum(model.probability(history + symbol) for symbol in model.support) - 1) < 1e-9, (order, repr(history))
        assert math.isfinite(model.log_probability('a' + UNK, order))
    assert model.log_probability('ab', 4) > model.log_probability('cc', 4)
    model.close()
    assert tokenize('Hola HOLA', 'word') == ['hola', 'hola']
    assert tokenize('Hola', 'char')[0] == 'H'
    print('Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.')

test_modified_kn()

Pruebas MKN: tres descuentos, normalizacion, continuaciones, BOS/EOS, UNK y capitalizacion correctos.


In [4]:
timing_rows, discount_rows, reductions = [], [], []

def fit_score_level(fit_texts, fit_labels, app_texts, level, orders, phase, fold):
    started = time.perf_counter()
    mapping = make_encoder(fit_texts, level)
    encoded_fit = [encode_text(text, level, mapping) for text in fit_texts]
    encoded_app = [encode_text(text, level, mapping) for text in app_texts]
    preprocessing = time.perf_counter() - started
    matrices = {order: np.zeros((len(app_texts), 6), dtype=np.float64) for order in orders}
    fit_seconds, score_seconds = 0.0, {order: 0.0 for order in orders}
    for class_index, label in enumerate(CLASSES):
        fit_started = time.perf_counter()
        model = ModifiedKN(max(orders), set(mapping.values())).fit(sequence for sequence, target in zip(encoded_fit, fit_labels) if target == class_index)
        fit_seconds += time.perf_counter() - fit_started
        discount_rows.extend(dict(fase=phase, pliegue=fold, nivel=level, clase=str(label), **record) for record in model.discount_records)
        for order in orders:
            score_started = time.perf_counter()
            matrices[order][:, class_index] = [model.log_probability(sequence, order) for sequence in encoded_app]
            score_seconds[order] += time.perf_counter() - score_started
        model.close()
        del model
        gc.collect()
        print(phase, fold, level, label, 'listo', round(time.perf_counter() - started, 1), 's', flush=True)
    times = {}
    for order in orders:
        times[order] = score_seconds[order] + (fit_seconds + preprocessing) / len(orders)
        timing_rows.append(dict(fase=phase, pliegue=fold, nivel=level, orden=order, ajuste_compartido_s=fit_seconds + preprocessing, puntuacion_s=score_seconds[order], segundos_amortizados=times[order]))
    return matrices, times

def combine_scores(char_scores, word_scores, priors, weight, mode, lengths):
    scores = weight * char_scores + (1 - weight) * word_scores + priors
    return scores / lengths[:, None] if mode == 'longitud' else scores

def macro_fast(targets, predictions):
    matrix = np.bincount(targets * 6 + predictions, minlength=36).reshape(6, 6)
    denominator = matrix.sum(axis=0) + matrix.sum(axis=1)
    return float(np.divide(2 * matrix.diagonal(), denominator, out=np.zeros(6), where=denominator != 0).mean())

def calibrate_bias(scores, targets):
    bias = np.zeros(6)
    baseline = macro_fast(targets, scores.argmax(axis=1))
    current = baseline
    sorted_scores = np.sort(scores, axis=1)
    scale = max(1e-4, min(20.0, float(np.median(sorted_scores[:, -1] - sorted_scores[:, -2]))))
    for step in [scale * 2, scale, scale / 2, scale / 4]:
        for class_index in range(6):
            adjusted = scores + bias
            adjusted[:, class_index] = -np.inf
            alternatives = adjusted.argmax(axis=1)
            alternative_max = adjusted[np.arange(len(targets)), alternatives]
            best_value = bias[class_index]
            for value in bias[class_index] + step * np.array([-2, -1, -0.5, 0, 0.5, 1, 2]):
                candidate = scores[:, class_index] + value
                choose = (candidate > alternative_max) | ((candidate == alternative_max) & (class_index < alternatives))
                predictions = np.where(choose, class_index, alternatives)
                result = macro_fast(targets, predictions)
                if result > current + 1e-12:
                    current, best_value = result, float(value)
            bias[class_index] = best_value
        bias -= bias.mean()
    return bias, baseline, macro_fast(targets, (scores + bias).argmax(axis=1))

check_targets = np.array([0, 1, 2, 3, 4, 5, 5])
check_predictions = np.array([0, 0, 2, 3, 5, 5, 4])
assert abs(macro_fast(check_targets, check_predictions) - f1_score(check_targets, check_predictions, labels=np.arange(6), average='macro', zero_division=0)) < 1e-12
check_scores = np.random.default_rng(SEED).normal(size=(len(check_targets), 6))
check_bias, check_before, check_after = calibrate_bias(check_scores, check_targets)
assert check_after >= check_before and abs(check_bias.mean()) < 1e-10
print('F1 y busqueda por coordenadas verificados.')

F1 y busqueda por coordenadas verificados.


## Seleccion dentro de desarrollo

Piloto: 10 % de los grupos de desarrollo, estratificados, con otro split 80/20 de esos grupos para medir. No se usa holdout. Si el coste amortizado de una configuracion supera 1200 s se reduce la rejilla y se informa. Las estimaciones de CV no sustituyen las mediciones reales.

CV: StratifiedKFold(5, shuffle=True, random_state=42) sobre grupos de desarrollo, luego expansion a sus filas. Se aprende vocabulario, conteos, descuentos y priors de nuevo en cada ajuste. Se comparan 4 x 3 x 5 x 2 configuraciones. El sesgo se ajusta por coordenadas sobre los scores OOF: cuatro pasadas de escala decreciente y siete desplazamientos por clase, aceptando solo mejoras. Se centra b. Es una busqueda local, no un optimo global garantizado.

Se reporta F1 OOF antes y despues: el segundo es optimista por calibracion y seleccion con las mismas etiquetas OOF. Ninguno utiliza holdout. La evaluacion independiente es el holdout final. En empates se prefieren menores ordenes y lambda, despues bruto. Los tiempos de entrenamiento son compartidos entre configuraciones y no deben sumarse como ejecuciones independientes.

In [5]:
pilot_groups, _ = train_test_split(dev_groups, train_size=0.1, stratify=group_targets[dev_groups], random_state=SEED)
pilot_fit_groups, pilot_app_groups = train_test_split(pilot_groups, test_size=0.2, stratify=group_targets[pilot_groups], random_state=SEED)
pilot_fit = np.flatnonzero(np.isin(groups_development, pilot_fit_groups))
pilot_app = np.flatnonzero(np.isin(groups_development, pilot_app_groups))
assert not set(texts_development[pilot_fit]) & set(texts_development[pilot_app])
active_orders, pilot_times, pilot_matrices = {}, {}, {}
for level, orders in [('char', CHAR_ORDERS), ('word', WORD_ORDERS)]:
    pilot_matrices[level], pilot_times[level] = fit_score_level(texts_development[pilot_fit], y_development[pilot_fit], texts_development[pilot_app], level, orders, 'piloto_10pct', 0)
    active = [order for order in orders if pilot_times[level][order] <= TIME_LIMIT]
    active_orders[level] = active or [min(orders, key=lambda order: pilot_times[level][order])]
    if active_orders[level] != orders:
        reductions.append(f'Piloto: {level} reducido a {active_orders[level]} por limite de 1200 s.')
pilot_prior = np.log(np.bincount(y_development[pilot_fit], minlength=6) / len(pilot_fit))
pilot_config_rows = []
for char_order, word_order, weight, mode in product(active_orders['char'], active_orders['word'], LAMBDAS, SCORE_MODES):
    started = time.perf_counter()
    scores = combine_scores(pilot_matrices['char'][char_order], pilot_matrices['word'][word_order], pilot_prior, weight, mode, lengths_development[pilot_app])
    _, before, after = calibrate_bias(scores, y_development[pilot_app])
    pilot_config_rows.append(dict(orden_car=char_order, orden_pal=word_order, lambda_car=weight, score=mode, f1_antes=before, f1_despues=after, segundos=pilot_times['char'][char_order] + pilot_times['word'][word_order] + time.perf_counter() - started))
pilot_table = pd.DataFrame(pilot_config_rows)
allowed_configurations = set(tuple(row) for row in pilot_table.loc[pilot_table.segundos <= TIME_LIMIT, ['orden_car', 'orden_pal', 'lambda_car', 'score']].itertuples(index=False, name=None))
if not allowed_configurations:
    fastest = pilot_table.sort_values('segundos').iloc[0]
    allowed_configurations = {(int(fastest.orden_car), int(fastest.orden_pal), float(fastest.lambda_car), fastest.score)}
if len(allowed_configurations) < len(pilot_table):
    reductions.append(f'Piloto: configuraciones reducidas de {len(pilot_table)} a {len(allowed_configurations)} por tiempo >1200 s.')
print('Piloto:', len(pilot_groups), 'grupos;', len(pilot_fit), 'filas ajuste;', len(pilot_app), 'filas validacion')
display(pd.DataFrame(timing_rows))
print(pilot_table.to_string(index=False))
print('Reducciones:', reductions or 'ninguna')
del pilot_matrices
gc.collect()

piloto_10pct 0 char 13-17 listo 0.9 s


piloto_10pct 0 char 18-24 listo 1.7 s


piloto_10pct 0 char 25-34 listo 2.5 s


piloto_10pct 0 char 35-49 listo 3.3 s


piloto_10pct 0 char 50-64 listo 4.1 s


...

piloto_10pct 0 word 65-xx listo 0.7 s


Piloto: 12623 grupos; 10580 filas ajuste; 2610 filas validacion


,fase,pliegue,nivel,orden,ajuste_compartido_s,puntuacion_s,segundos_amortizados
0,piloto_10pct,0,char,3,1.400863,0.231910,0.582126
1,piloto_10pct,0,char,4,1.400863,0.548316,0.898532
2,piloto_10pct,0,char,5,1.400863,0.981264,1.331480
3,piloto_10pct,0,char,6,1.400863,1.454677,1.804892
4,piloto_10pct,0,word,1,0.223484,0.046912,0.121407
5,piloto_10pct,0,word,2,0.223484,0.108814,0.183309
6,piloto_10pct,0,word,3,0.223484,0.219044,0.293539


 orden_car  orden_pal  lambda_car    score  f1_antes  f1_despues  segundos
         3          1        0.00    bruto  0.295792    0.315920  0.707607
         3          1        0.00 longitud  0.295792    0.322237  0.707922
         3          1        0.25    bruto  0.329943    0.335073  0.707626
         3          1        0.25 longitud  0.329943    0.338283  0.707974
         3          1        0.50    bruto  0.329617    0.341603  0.707515
         3          1        0.50 longitud  0.329617    0.340331  0.707756
         3          1        0.75    bruto  0.349051    0.356199  0.707440
         3          1        0.75 longitud  0.349051    0.355052  0.707762
         3          1        1.00    bruto  0.351902    0.360898  0.707626
         3          1        1.00 longitud  0.351902    0.360877  0.707824
         3          2        0.00    bruto  0.322015    0.326995  0.769356
         3          2        0.00 longitud  0.322015    0.326811  0.769676
         3          2    

0

In [6]:
cv_started = time.perf_counter()
group_folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED).split(dev_groups, group_targets[dev_groups]))
folds = []
for fit_groups_pos, app_groups_pos in group_folds:
    fit_idx = np.flatnonzero(np.isin(groups_development, dev_groups[fit_groups_pos]))
    app_idx = np.flatnonzero(np.isin(groups_development, dev_groups[app_groups_pos]))
    assert not set(texts_development[fit_idx]) & set(texts_development[app_idx])
    folds.append((fit_idx, app_idx))
oof_char = {order: np.zeros((len(y_development), 6)) for order in active_orders['char']}
oof_word = {order: np.zeros((len(y_development), 6)) for order in active_orders['word']}
oof_prior = np.zeros((len(y_development), 6))
component_times = {'char': Counter(), 'word': Counter()}
for fold_number, (fit_idx, app_idx) in enumerate(folds, 1):
    counts = np.bincount(y_development[fit_idx], minlength=6)
    assert (counts > 0).all()
    oof_prior[app_idx] = np.log(counts / counts.sum())
    for level, storage in [('char', oof_char), ('word', oof_word)]:
        scores, measured = fit_score_level(texts_development[fit_idx], y_development[fit_idx], texts_development[app_idx], level, active_orders[level], 'CV', fold_number)
        for order in active_orders[level]:
            storage[order][app_idx] = scores[order]
            component_times[level][order] += measured[order]
        del scores
        gc.collect()
    print('Pliegue', fold_number, 'completo; interseccion normalizada 0; CV', round(time.perf_counter() - cv_started, 1), 's', flush=True)
search_rows, bias_vectors = [], {}
for char_order, word_order, weight, mode in product(active_orders['char'], active_orders['word'], LAMBDAS, SCORE_MODES):
    key = (char_order, word_order, weight, mode)
    if key not in allowed_configurations:
        continue
    started = time.perf_counter()
    scores = combine_scores(oof_char[char_order], oof_word[word_order], oof_prior, weight, mode, lengths_development)
    bias, before, after = calibrate_bias(scores, y_development)
    predictions = (scores + bias).argmax(axis=1)
    fold_f1 = [macro_fast(y_development[app_idx], predictions[app_idx]) for _, app_idx in folds]
    elapsed = time.perf_counter() - started
    bias_vectors[key] = bias
    search_rows.append(dict(orden_car=char_order, orden_pal=word_order, lambda_car=weight, score=mode, f1_antes=before, f1_despues=after, f1_pliegues_media=float(np.mean(fold_f1)), f1_pliegues_std=float(np.std(fold_f1)), componentes_cv_s=component_times['char'][char_order] + component_times['word'][word_order], calibracion_s=elapsed))
results = pd.DataFrame(search_rows).sort_values(['f1_despues', 'orden_car', 'orden_pal', 'lambda_car', 'score'], ascending=[False, True, True, True, True]).reset_index(drop=True)
best = results.iloc[0]
best_char, best_word, best_weight, best_mode = int(best.orden_car), int(best.orden_pal), float(best.lambda_car), best.score
best_bias = bias_vectors[(best_char, best_word, best_weight, best_mode)].copy()
print('TABLA_CV_COMPLETA')
print(results.sort_values(['score', 'orden_car', 'orden_pal', 'lambda_car']).to_string(index=False))
print('MEJOR', best_char, best_word, best_weight, best_mode, '| Sesgos', best_bias.tolist())
print('Tiempo CV y seleccion', round(time.perf_counter() - cv_started, 1), 's')
print('Reducciones', reductions or 'ninguna')
display(pd.DataFrame(timing_rows))
display(pd.DataFrame(discount_rows).groupby(['nivel', 'origen', 'orden'])[['D1', 'D2', 'D3']].agg(['min', 'mean', 'max']))

CV 1 char 13-17 listo 8.1 s


CV 1 char 18-24 listo 15.7 s


CV 1 char 25-34 listo 23.0 s


CV 1 char 35-49 listo 30.7 s


CV 1 char 50-64 listo 39.0 s


...

CV 5 word 65-xx listo 8.3 s


Pliegue 5 completo; interseccion normalizada 0; CV 260.4 s


TABLA_CV_COMPLETA
 orden_car  orden_pal  lambda_car    score  f1_antes  f1_despues  f1_pliegues_media  f1_pliegues_std  componentes_cv_s  calibracion_s
         3          1        0.00    bruto  0.356088    0.370264           0.370205         0.001515         31.401561       0.145551
         3          1        0.25    bruto  0.393269    0.400283           0.400223         0.002741         31.401561       0.145062
         3          1        0.50    bruto  0.404183    0.408703           0.408657         0.003567         31.401561       0.145518
         3          1        0.75    bruto  0.411230    0.414241           0.414214         0.003384         31.401561       0.142834
         3          1        1.00    bruto  0.398905    0.403656           0.403573         0.004798         31.401561       0.141068
         3          2        0.00    bruto  0.401977    0.403750           0.403670         0.003875         36.559814       0.145119
         3          2        0.25    bruto  

,fase,pliegue,nivel,orden,ajuste_compartido_s,puntuacion_s,segundos_amortizados
0,piloto_10pct,0,char,3,1.400863,0.231910,0.582126
1,piloto_10pct,0,char,4,1.400863,0.548316,0.898532
2,piloto_10pct,0,char,5,1.400863,0.981264,1.331480
3,piloto_10pct,0,char,6,1.400863,1.454677,1.804892
4,piloto_10pct,0,word,1,0.223484,0.046912,0.121407
5,piloto_10pct,0,word,2,0.223484,0.108814,0.183309
6,piloto_10pct,0,word,3,0.223484,0.219044,0.293539
7,CV,1,char,3,13.261444,2.066948,5.382309
8,CV,1,char,4,13.261444,4.804902,8.120263
9,CV,1,char,5,13.261444,8.768984,12.084345


D1                            D2            \
                                 min      mean       max       min      mean   
nivel origen         orden                                                     
char  continuaciones 2      0.507503  0.532060  0.571885  0.913933  1.058607   
                     3      0.600886  0.613763  0.647757  1.015624  1.055106   
                     4      0.663817  0.676210  0.710858  1.043099  1.090140   
                     5      0.718882  0.737938  0.798766  1.117210  1.170058   
      frecuencias    2      0.479029  0.521474  0.560538  0.712485  1.003667   
                     3      0.572497  0.587779  0.617839  0.957421  1.029897   
                     4      0.622097  0.636445  0.664078  0.998626  1.038464   
                     5      0.659883  0.681270  0.727548  1.012201  1.053839   
                     6      0.696931  0.725752  0.781883  1.054138  1.084373   
word  continuaciones 2      0.788981  0.808859  0.848770  1.161009  1.206729   
      frecuencias    2      0.757382  0.779834  0.825574  1.112353  1.187928   
                     3      0.872356  0.889634  0.918767  1.111325  1.255086   

                                            D3                      
                                 max       min      mean       max  
nivel origen         orden                                          
char  continuaciones 2      1.170108  1.008926  1.493272  1.725826  
                     3      1.112968  1.325627  1.414619  1.527276  
                     4      1.175812  1.366360  1.442885  1.586828  
                     5      1.267698  1.469595  1.521912  1.632581  
      frecuencias    2      1.158728  1.242638  1.505670  1.805494  
                     3      1.079361  1.182494  1.434974  1.560948  
                     4      1.111171  1.299858  1.412966  1.470995  
                     5      1.113118  1.330054  1.400539  1.458449  
                     6      1.203740  1.314418  1.390202  1.537902  
word  continuaciones 2      1.375095  1.250533  1.457962  1.613658  
      frecuencias    2      1.356518  1.288843  1.438357  1.560538  
                     3      1.408835  0.943235  1.351385  1.508350

## Holdout reservado

Ordenes, peso, variante y b ya estan congelados. Este holdout por grupos no es el de v5 ni el historico de Kelly: las diferencias frente a 0.44945 y 0.6075 son descriptivas, no mejoras emparejadas certificadas. v3 reporta 0.61765 de Kaggle; el dato v4 de Kaggle 0.62832 fue aportado por el usuario. No se comparan esos scores de Kaggle directamente con este holdout. Las diez interpretaciones de errores son hipotesis, no explicaciones causales verificadas.

In [7]:
assert 'holdout_f1' not in globals(), 'Holdout ya evaluado: no reutilizar para seleccionar.'
holdout_started = time.perf_counter()
texts_holdout = train.normalized.iloc[idx_holdout].to_numpy()
lengths_holdout = np.array([max(1, len(text)) for text in texts_holdout], dtype=float)
char_holdout, _ = fit_score_level(texts_development, y_development, texts_holdout, 'char', [best_char], 'holdout', 0)
word_holdout, _ = fit_score_level(texts_development, y_development, texts_holdout, 'word', [best_word], 'holdout', 0)
prior_development = np.log(np.bincount(y_development, minlength=6) / len(y_development))
holdout_scores = combine_scores(char_holdout[best_char], word_holdout[best_word], prior_development, best_weight, best_mode, lengths_holdout) + best_bias
holdout_pred = holdout_scores.argmax(axis=1)
y_holdout = y_all[idx_holdout]
holdout_f1 = f1_score(y_holdout, holdout_pred, labels=np.arange(6), average='macro', zero_division=0)
report = classification_report(y_holdout, holdout_pred, labels=np.arange(6), target_names=CLASSES, output_dict=True, zero_division=0)
class_table = pd.DataFrame(report).T.loc[CLASSES, ['precision', 'recall', 'f1-score', 'support']]
print('F1 MACRO HOLDOUT', holdout_f1)
display(class_table)
display(pd.DataFrame(confusion_matrix(y_holdout, holdout_pred, labels=np.arange(6)), index=pd.Index(CLASSES, name='real'), columns=pd.Index(CLASSES, name='predicha')))
display(pd.DataFrame({'referencia': ['v5 historico', 'v3 historico'], 'f1_holdout_publicado': [0.44945, 0.6075], 'diferencia_v6': [holdout_f1 - 0.44945, holdout_f1 - 0.6075]}))

def interpret_error(text, real, predicted):
    if text.strip() == '<url>':
        return 'Solo URL: no hay contenido lexico para distinguir edades.'
    if len(TOKEN_RE.findall(text)) < 5:
        return 'Muy pocas palabras: senal linguistica escasa; hipotesis.'
    if abs(real - predicted) == 1:
        return 'Rangos contiguos: pueden compartir vocabulario y estilo.'
    if '<user>' in text:
        return 'La sustitucion de cuentas elimina una posible pista de comunidad.'
    return 'El contenido puede ser comun a varias edades; requiere revision humana.'

error_positions = np.flatnonzero(y_holdout != holdout_pred)[:10]
error_table = pd.DataFrame([dict(texto=train.text.iloc[idx_holdout[position]], real=CLASSES[y_holdout[position]], predicha=CLASSES[holdout_pred[position]], interpretacion=interpret_error(texts_holdout[position], int(y_holdout[position]), int(holdout_pred[position]))) for position in error_positions])
with pd.option_context('display.max_colwidth', 180):
    display(error_table)
print('Tiempo holdout', round(time.perf_counter() - holdout_started, 1), 's')
del char_holdout, word_holdout
gc.collect()

holdout 0 char 13-17 listo 2.1 s


holdout 0 char 18-24 listo 3.4 s


holdout 0 char 25-34 listo 4.7 s


holdout 0 char 35-49 listo 6.1 s


holdout 0 char 50-64 listo 7.5 s


...

predicha,13-17,18-24,25-34,35-49,50-64,65-xx
real,,,,,,
13-17,2512,1531,1154,372,206,151
18-24,1046,2288,1670,470,269,256
25-34,497,1152,2586,777,495,418
35-49,254,590,1721,2013,694,735
50-64,138,329,1526,785,2524,799
65-xx,139,286,1446,604,752,2931


,referencia,f1_holdout_publicado,diferencia_v6
0,v5 historico,0.44945,-0.030984
1,v3 historico,0.60750,-0.189034


,texto,real,predicha,interpretacion
0,http://t.co/Jp4hFcwMF6,13-17,25-34,Solo URL: no hay contenido lexico para distinguir edades.
1,https://t.co/Q4UxtLkSUa,65-xx,25-34,Solo URL: no hay contenido lexico para distinguir edades.
2,hope http://t.co/1CYf4P8PD5,18-24,13-17,Muy pocas palabras: senal linguistica escasa; hipotesis.
3,https://t.co/de3OaDIih0,50-64,25-34,Solo URL: no hay contenido lexico para distinguir edades.
4,RT @NoticiasCaracol: Estallan tres nuevos escandalos en la Policia Nacional https://t.co/LhJXp4zG8F https://t.co/eOo8ikcjMV,25-34,35-49,Rangos contiguos: pueden compartir vocabulario y estilo.
5,"RT @zachallia: If you use Instagram & Google Chrome and you aren't using Instagram for Chrome, you're real silly: http://t.co/pUVCNINGET",13-17,25-34,La sustitucion de cuentas elimina una posible pista de comunidad.
6,Es lo que todos hacemos. Tambien para desenamorarnos https://t.co/ridGAGXfpY,50-64,25-34,El contenido puede ser comun a varias edades; requiere revision humana.
7,@Padreaicardo buenos Dias le pido por favor que me ayude a Orar por la salud de mis familiares y que se mejoren las ventas en mi negocio.,25-34,18-24,Rangos contiguos: pueden compartir vocabulario y estilo.
8,RT @PrensaMCM: #ATENCION Manana a las 10 am @VenteVenezuela y @MariaCorinaYa responderan en rueda de prensa a las falsas acusaciones del Mi,65-xx,50-64,Rangos contiguos: pueden compartir vocabulario y estilo.
9,Voy a escuchar a Nereo http://t.co/NELxnhQBoW,50-64,18-24,El contenido puede ser comun a varias edades; requiere revision humana.


Tiempo holdout 16.9 s


0

In [8]:
final_started = time.perf_counter()
verify_csv(DATA_DIR / 'age_test.csv', ['id', 'text'], ';', True)
test = pd.read_csv(DATA_DIR / 'age_test.csv', sep=';', encoding='utf-8-sig', dtype=str, keep_default_na=False)
assert list(test.columns) == ['id', 'text'] and test.id.is_unique and test.id.ne('').all()
test_texts = test.text.fillna('').str.strip().map(html.unescape).map(normalize_text).to_numpy()
test_lengths = np.array([max(1, len(text)) for text in test_texts], dtype=float)
print('Test filas', len(test), '| Solo URL', int(sum(text.strip() == '<url>' for text in test_texts)))
char_final, _ = fit_score_level(train.normalized.to_numpy(), y_all, test_texts, 'char', [best_char], 'final', 0)
word_final, _ = fit_score_level(train.normalized.to_numpy(), y_all, test_texts, 'word', [best_word], 'final', 0)
prior_final = np.log(np.bincount(y_all, minlength=6) / len(y_all))
final_scores = combine_scores(char_final[best_char], word_final[best_word], prior_final, best_weight, best_mode, test_lengths) + best_bias
submission = pd.DataFrame({'id': test.id, 'age_range': CLASSES[final_scores.argmax(axis=1)]})
assert list(submission.columns) == reference_columns == ['id', 'age_range']
assert len(submission) == len(test) and submission.id.is_unique and submission.id.tolist() == test.id.tolist()
assert submission.age_range.isin(CLASSES).all()
with SUB_PATH.open('x', encoding='utf-8', newline='') as handle:
    submission.to_csv(handle, sep=',', index=False)
with SUB_PATH.open('rb') as handle:
    prefix = handle.read(64)
assert not prefix.startswith(b'\xef\xbb\xbf') and prefix.splitlines()[0] == b'id,age_range'
assert INPUT_HASHES == {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest() for name in INPUT_HASHES}
print('FILAS TEST/SALIDA', len(test), len(submission), '| COINCIDEN', len(test) == len(submission))
print('Tiempo final', round(time.perf_counter() - final_started, 1), 's')
print('ARCHIVOS', NOTEBOOK_PATH, SUB_PATH, SPLIT_PATH, OUT_DIR / 'requirements.txt', sep='\n')
display(submission.head())
display(submission.age_range.value_counts().reindex(CLASSES).rename('predicciones').to_frame())
display(pd.DataFrame(timing_rows))
display(pd.DataFrame(discount_rows).loc[lambda frame: frame.fase == 'final'])

age_test.csv BOM= True separador= ';' columnas= ['id', 'text']


Test filas 112892 | Solo URL 2438


final 0 char 13-17 listo 4.2 s


final 0 char 18-24 listo 7.4 s


final 0 char 25-34 listo 10.8 s


...

FILAS TEST/SALIDA 112892 112892 | COINCIDEN True
Tiempo final 42.8 s
ARCHIVOS
./workshops/Kaggle-tweet-age/TweetAge_v6_charLM_kn.ipynb
./workshops/Kaggle-tweet-age/age_submission_charLM_kn_v6.csv
./workshops/Kaggle-tweet-age/split_idx_v6_seed42.npz
./workshops/Kaggle-tweet-age/requirements.txt


,id,age_range
0,1,65-xx
1,2,13-17
2,3,13-17
3,4,65-xx
4,5,65-xx


,predicciones
age_range,
13-17,15730
18-24,21187
25-34,23442
35-49,17359
50-64,17008
65-xx,18166


,fase,pliegue,nivel,orden,ajuste_compartido_s,puntuacion_s,segundos_amortizados
0,piloto_10pct,0,char,3,1.400863,0.231910,0.582126
1,piloto_10pct,0,char,4,1.400863,0.548316,0.898532
2,piloto_10pct,0,char,5,1.400863,0.981264,1.331480
3,piloto_10pct,0,char,6,1.400863,1.454677,1.804892
4,piloto_10pct,0,word,1,0.223484,0.046912,0.121407
5,piloto_10pct,0,word,2,0.223484,0.108814,0.183309
6,piloto_10pct,0,word,3,0.223484,0.219044,0.293539
7,CV,1,char,3,13.261444,2.066948,5.382309
8,CV,1,char,4,13.261444,4.804902,8.120263
9,CV,1,char,5,13.261444,8.768984,12.084345


,fase,pliegue,nivel,clase,origen,orden,D1,D2,D3,N1,N2,N3,N4
480,final,0,char,13-17,frecuencias,2,0.498147,1.066762,1.566144,941,474,296,213
481,final,0,char,13-17,frecuencias,3,0.581780,1.019195,1.482195,12798,4600,2585,1686
482,final,0,char,13-17,frecuencias,4,0.630691,1.041800,1.393681,60301,17655,8941,5693
483,final,0,char,13-17,continuaciones,2,0.510276,1.173662,1.517052,1440,691,373,271
484,final,0,char,13-17,continuaciones,3,0.607715,1.027443,1.432755,17738,5725,3054,1969
485,final,0,char,18-24,frecuencias,2,0.513627,1.060196,1.519012,980,464,283,204
486,final,0,char,18-24,frecuencias,3,0.582067,1.029063,1.531811,13100,4703,2615,1649
487,final,0,char,18-24,frecuencias,4,0.637470,1.020155,1.426167,61308,17433,8932,5513
488,final,0,char,18-24,continuaciones,2,0.516788,1.098353,1.630847,1416,662,385,255
489,final,0,char,18-24,continuaciones,3,0.604721,1.062033,1.468285,17523,5727,2961,1875


In [9]:
worst_class = class_table['f1-score'].idxmin()
summary_text = f'''## Resumen de resultados v6

Configuracion: caracteres **{best_char}**, palabras **{best_word}**, lambda **{best_weight}**, score **{best_mode}**. F1 OOF antes/despues de calibrar: **{float(best.f1_antes):.5f} / {float(best.f1_despues):.5f}**. La mejora de ajuste por b no es una estimacion independiente; b se calibro sobre esas etiquetas OOF. Descuentos D1/D2/D3 estimados solo en el ajuste de cada modelo; se muestran sus tablas y fallbacks.

F1 macro holdout: **{holdout_f1:.5f}**. Diferencia descriptiva con v5: **{holdout_f1 - 0.44945:+.5f}**; con v3: **{holdout_f1 - 0.6075:+.5f}**. Los splits son distintos: este evita textos normalizados identicos entre lados. No es una comparacion emparejada y no se confunde con Kaggle.

Desarrollo: **{len(dev_groups)} grupos / {len(idx_development)} filas**. Holdout: **{len(holdout_groups)} grupos / {len(idx_holdout)} filas**. Interseccion: **{len(intersection)}**. En CV tambien se verifico cero en cada pliegue.

Peor clase: **{worst_class}**, F1 **{class_table.loc[worst_class, 'f1-score']:.5f}**. La confusion con otras edades y el contenido escaso son posibles explicaciones, no causas probadas. Se compararon scores brutos y por longitud solo en desarrollo; no se declara causalidad a partir de F1 por clase.

Sesgos en orden de edad: `{best_bias.tolist()}`.

Test y salida: **{len(test)} / {len(submission)} filas**, iguales, con IDs en el mismo orden. Entorno dentro del proyecto, versiones fijadas en requirements.txt. Reducciones: **{'; '.join(reductions) if reductions else 'ninguna'}**.

Antes de Kaggle: revisar con el equipo rendimiento por clase, errores y reglas; decidir si interesa esta linea base permitida sin anunciar una mejora frente a modelos con otro split. No se envio nada a Kaggle ni se hicieron commits o push. El dato de v4 Kaggle 0.62832 proviene del usuario, no de una consulta independiente.
'''
print('TABLA FINAL OOF ANTES Y DESPUES')
print(results.sort_values(['score', 'orden_car', 'orden_pal', 'lambda_car']).to_string(index=False))
display(Markdown(summary_text))

TABLA FINAL OOF ANTES Y DESPUES
 orden_car  orden_pal  lambda_car    score  f1_antes  f1_despues  f1_pliegues_media  f1_pliegues_std  componentes_cv_s  calibracion_s
         3          1        0.00    bruto  0.356088    0.370264           0.370205         0.001515         31.401561       0.145551
         3          1        0.25    bruto  0.393269    0.400283           0.400223         0.002741         31.401561       0.145062
         3          1        0.50    bruto  0.404183    0.408703           0.408657         0.003567         31.401561       0.145518
         3          1        0.75    bruto  0.411230    0.414241           0.414214         0.003384         31.401561       0.142834
         3          1        1.00    bruto  0.398905    0.403656           0.403573         0.004798         31.401561       0.141068
         3          2        0.00    bruto  0.401977    0.403750           0.403670         0.003875         36.559814       0.145119
         3          2        0

## Resumen de resultados v6

Configuracion: caracteres **4**, palabras **3**, lambda **0.75**, score **bruto**. F1 OOF antes/despues de calibrar: **0.44106 / 0.44353**. La mejora de ajuste por b no es una estimacion independiente; b se calibro sobre esas etiquetas OOF. Descuentos D1/D2/D3 estimados solo en el ajuste de cada modelo; se muestran sus tablas y fallbacks.

F1 macro holdout: **0.41847**. Diferencia descriptiva con v5: **-0.03098**; con v3: **-0.18903**. Los splits son distintos: este evita textos normalizados identicos entre lados. No es una comparacion emparejada y no se confunde con Kaggle.

Desarrollo: **126232 grupos / 133065 filas**. Holdout: **31558 grupos / 36116 filas**. Interseccion: **0**. En CV tambien se verifico cero en cada pliegue.

Peor clase: **25-34**, F1 **0.32269**. La confusion con otras edades y el contenido escaso son posibles explicaciones, no causas probadas. Se compararon scores brutos y por longitud solo en desarrollo; no se declara causalidad a partir de F1 por clase.

Sesgos en orden de edad: `[-0.050482373487580424, 0.252411867437902, 0.5553061083633846, 0.25241186743790206, -0.6562708553385452, -0.3533766144130629]`.

Test y salida: **112892 / 112892 filas**, iguales, con IDs en el mismo orden. Entorno dentro del proyecto, versiones fijadas en requirements.txt. Reducciones: **ninguna**.

Antes de Kaggle: revisar con el equipo rendimiento por clase, errores y reglas; decidir si interesa esta linea base permitida sin anunciar una mejora frente a modelos con otro split. No se envio nada a Kaggle ni se hicieron commits o push. El dato de v4 Kaggle 0.62832 proviene del usuario, no de una consulta independiente.


## Resumen final observado

Se ejecutaron **120 configuraciones** sin reducir la rejilla. Gano **caracteres 4, palabras 3, lambda=0.75, score bruto**. F1 OOF antes/despues de calibrar: **0.44106 / 0.44353**. Es una mejora de ajuste, no una estimacion independiente, porque b se calibro con esas etiquetas OOF. La variante por longitud no fue elegida. CV y seleccion: **278.1 s**; holdout: **16.9 s**; ajuste y prediccion final: **42.8 s**.

**F1 macro holdout: 0.4184663**. Diferencias descriptivas: **-0.0309837 frente a v5 (0.44945)** y **-0.1890337 frente a v3 (0.6075)**. Los splits son distintos; no son comparaciones emparejadas ni se comparan con Kaggle. El score de v4 Kaggle 0.62832 proviene del usuario y no fue verificado independientemente.

| Edad | F1 holdout |
|---|---:|
| 13-17 | 0.477930 |
| 18-24 | 0.375852 |
| 25-34 | 0.322685 |
| 35-49 | 0.365071 |
| 50-64 | 0.457205 |
| 65-xx | 0.512055 |

Desarrollo: **126232 grupos y 133065 filas**. Holdout: **31558 grupos y 36116 filas**. **Interseccion normalizada: 0**, tambien en los cinco pliegues internos. El 80/20 se aplica a grupos, no exactamente a filas. Los **680 grupos con etiquetas contradictorias** se estratificaron por mayoria con desempate por edad. Se excluyeron 155 filas de 169336 originales; quedaron **169181** utilizables y **3785** textos solo `<url>`.

La peor clase es **25-34**. La matriz y los errores muestran confusion entre edades y varios textos sin contenido lexico; son observaciones, no causas verificadas. La normalizacion de cuentas y URLs elimina posibles pistas.

Sesgos en orden de edad: **[-0.05048237, 0.25241187, 0.55530611, 0.25241187, -0.65627086, -0.35337661]**. Los descuentos D1, D2 y D3+ se estimaron por clase, orden y tipo de conteo, solo sobre el ajuste de cada modelo. Pasaron las pruebas de normalizacion y desconocidos y la guardia AST, incluida la prohibicion de sklearn.linear_model.

**Test y salida: 112892 / 112892 filas**, iguales, con IDs unicos y en el mismo orden, seis clases validas, coma y UTF-8 sin BOM. Se guardaron indices y SHA-256 en el NPZ. Los hashes de los archivos preexistentes revisados no cambiaron. El entorno esta en **.venv-v6 dentro de la carpeta del taller** y las dependencias estan fijadas en requirements.txt.

**Antes de Kaggle:** revisar con el equipo reglas, errores y cupo. No anunciar una mejora frente a modelos con otra particion; v6 elimina el solapamiento exacto y aporta una evaluacion mas estricta y reproducible. No se envio nada a Kaggle ni se ejecutaron commit o push.